In [1]:
%pip install --only-binary=:all: openpyxl==3.1.5

  Using cached openpyxl-3.1.5-py2.py3-none-any.whl.metadata (2.5 kB)
  Using cached et_xmlfile-2.0.0-py3-none-any.whl.metadata (2.7 kB)
Using cached openpyxl-3.1.5-py2.py3-none-any.whl (250 kB)
Using cached et_xmlfile-2.0.0-py3-none-any.whl (18 kB)
Note: you may need to restart the kernel to use updated packages.


In [2]:
import sys
import pandas as pd
from pathlib import Path

proyecto = Path(
    r"C:\Users\Juan Palacios\Documents\Universidad"
    r"\1_Semestre_Especializacion\GESTION_DE_DATOS"
    r"\Proyecto_GYA\ProyectoFinalETL"
)

bronze = proyecto / "data" / "bronze"
archivo_pob = bronze / "PPED-AreaMun-2018-2042_VP.xlsx"

print("Python:", sys.executable)
print("Existe:", archivo_pob.is_file())

with pd.ExcelFile(archivo_pob, engine="openpyxl") as libro:
    print("Hojas:", libro.sheet_names)

Python: c:\Users\Juan Palacios\Documents\Universidad\1_Semestre_Especializacion\GESTION_DE_DATOS\Proyecto_GYA\ProyectoFinalETL\venv\Scripts\python.exe
Existe: True
Hojas: ['Índice', 'PPED', 'PobMunicipalxÁrea']


In [3]:
hoja_original = pd.read_excel(
    archivo_pob,
    sheet_name="PobMunicipalxÁrea",
    header=7,
    dtype="string",
    engine="openpyxl",
    keep_default_na=False,
    na_values=[""],
)

campos_datos = ["MPIO", "AÑO", "ÁREA GEOGRÁFICA", "TOTAL"]

# Una fila tabular contiene información en alguno de estos campos.
es_dato = hoja_original[campos_datos].notna().any(axis=1)

print("NOTAS DEL ARCHIVO")
display(
    hoja_original.loc[~es_dato, ["DP"]].dropna()
)

# Conservar los valores originales y crear una copia de revisión.
pob_original = hoja_original.loc[es_dato].copy()

pob_revision = pob_original.apply(
    lambda columna: columna.str.strip().replace("", pd.NA)
)

print("Filas de datos:", len(pob_original))
print("Columnas:", len(pob_original.columns))

display(pob_original.head())

NOTAS DEL ARCHIVO


,DP
84230,"Fuente: DANE, Proyecciones de población - Dire..."
84231,Nota: Desde el 1 de diciembre de 2019 comenzó ...
84232,Nota: A partir del decreto 0284 del 26 de dic...
84233,Actualizado el 30 de Julio de 2025


Filas de datos: 84225
Columnas: 7


,DP,DPNOM,MPIO,DPMP,AÑO,ÁREA GEOGRÁFICA,TOTAL
1,05,Antioquia,05001,Medellín,2018,Cabecera Municipal,2382405
2,05,Antioquia,05001,Medellín,2018,Centros Poblados y Rural Disperso,44728
3,05,Antioquia,05001,Medellín,2018,Total,2427133
4,05,Antioquia,05002,Abejorral,2018,Cabecera Municipal,7669
5,05,Antioquia,05002,Abejorral,2018,Centros Poblados y Rural Disperso,12699


In [4]:
pob_revision.info()

calidad_pob = pd.DataFrame({
    "tipo": pob_revision.dtypes.astype(str),
    "faltantes": pob_revision.isna().sum(),
    "valores_distintos": pob_revision.nunique(),
})

calidad_pob["porcentaje_faltante"] = (
    calidad_pob["faltantes"] / len(pob_revision) * 100
).round(2)

display(calidad_pob)

numeros_pob = pob_revision[["AÑO", "TOTAL"]].apply(
    lambda columna: pd.to_numeric(columna, errors="coerce")
)

errores_conversion = (
    numeros_pob.isna()
    & pob_revision[["AÑO", "TOTAL"]].notna()
)

print("VALORES NO CONVERTIBLES")
display(
    errores_conversion.sum()
    .rename("no_convertibles")
    .to_frame()
)

display(
    pob_original.loc[errores_conversion.any(axis=1)]
)

<class 'pandas.core.frame.DataFrame'>
Index: 84225 entries, 1 to 84225
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   DP               84225 non-null  string
 1   DPNOM            84225 non-null  string
 2   MPIO             84225 non-null  string
 3   DPMP             84225 non-null  string
 4   AÑO              84225 non-null  string
 5   ÁREA GEOGRÁFICA  84225 non-null  string
 6   TOTAL            84225 non-null  string
dtypes: string(7)
memory usage: 5.1 MB


,tipo,faltantes,valores_distintos,porcentaje_faltante
DP,string,0,33,0.0
DPNOM,string,0,33,0.0
MPIO,string,0,1123,0.0
DPMP,string,0,1040,0.0
AÑO,string,0,25,0.0
ÁREA GEOGRÁFICA,string,0,3,0.0
TOTAL,string,0,34214,0.0


VALORES NO CONVERTIBLES


,no_convertibles
AÑO,0
TOTAL,0


,DP,DPNOM,MPIO,DPMP,AÑO,ÁREA GEOGRÁFICA,TOTAL


In [5]:
def normalizar_codigo(serie, longitud):
    texto = serie.astype("string").str.strip()

    valido = (
        texto.str.fullmatch(rf"[0-9]{{1,{longitud}}}", na=False)
        & ~texto.str.fullmatch(r"0+", na=False)
    )

    return texto.where(valido).str.zfill(longitud)


pob = pob_revision.copy()

pob["codigo_municipio"] = normalizar_codigo(pob["MPIO"], 5)
pob["codigo_departamento"] = normalizar_codigo(pob["DP"], 2)
pob["anio"] = numeros_pob["AÑO"]
pob["poblacion"] = numeros_pob["TOTAL"]

areas_esperadas = [
    "Cabecera Municipal",
    "Centros Poblados y Rural Disperso",
    "Total",
]

claves = ["codigo_municipio", "anio", "ÁREA GEOGRÁFICA"]

controles_pob = pd.Series({
    "filas_con_faltantes":
        pob_revision.isna().any(axis=1).sum(),

    "codigo_municipio_invalido":
        pob["codigo_municipio"].isna().sum(),

    "codigo_departamento_invalido":
        pob["codigo_departamento"].isna().sum(),

    "anio_invalido": (
        pob["anio"].isna()
        | ~pob["anio"].between(2018, 2042)
        | pob["anio"].mod(1).ne(0)
    ).sum(),

    "poblacion_no_numerica_o_faltante":
        pob["poblacion"].isna().sum(),

    "poblacion_negativa":
        pob["poblacion"].lt(0).sum(),

    "poblacion_fraccionaria": (
        pob["poblacion"].notna()
        & pob["poblacion"].mod(1).ne(0)
    ).sum(),

    "poblacion_cero":
        pob["poblacion"].eq(0).sum(),

    "area_no_reconocida": (
        ~pob["ÁREA GEOGRÁFICA"].isin(areas_esperadas)
    ).sum(),

    "filas_duplicadas":
        pob_original.duplicated().sum(),

    "claves_duplicadas":
        pob.duplicated(claves).sum(),
})

display(controles_pob.rename("cantidad").to_frame())

# Los primeros dos dígitos del código municipal deben ser
# iguales al código del departamento.
departamento_inconsistente = (
    pob["codigo_municipio"].str[:2]
    .ne(pob["codigo_departamento"])
    .fillna(True)
)

print(
    "Departamento inconsistente:",
    departamento_inconsistente.sum(),
)

display(
    pob.loc[departamento_inconsistente].head(20)
)

,cantidad
filas_con_faltantes,0
codigo_municipio_invalido,0
codigo_departamento_invalido,0
anio_invalido,0
poblacion_no_numerica_o_faltante,0
poblacion_negativa,0
poblacion_fraccionaria,0
poblacion_cero,541
area_no_reconocida,0
filas_duplicadas,0


Departamento inconsistente: 0


,DP,DPNOM,MPIO,DPMP,AÑO,ÁREA GEOGRÁFICA,TOTAL,codigo_municipio,codigo_departamento,anio,poblacion


In [6]:
print("REGISTROS POR AÑO")
display(
    pob.groupby("anio").agg(
        registros=("MPIO", "size"),
        codigos_territoriales=("codigo_municipio", "nunique"),
        areas=("ÁREA GEOGRÁFICA", "nunique"),
    )
)

print("REGISTROS POR ÁREA")
display(
    pob["ÁREA GEOGRÁFICA"]
    .value_counts()
    .to_frame("registros")
)

print("RANGOS DE POBLACIÓN")
display(
    pob.groupby("ÁREA GEOGRÁFICA")["poblacion"]
    .agg(["count", "min", "max"])
)

print("CEROS POR ÁREA")
display(
    pob.loc[pob["poblacion"].eq(0)]
    .groupby("ÁREA GEOGRÁFICA")
    .size()
    .to_frame("registros")
)

print("POBLACIÓN TOTAL EN CERO")
display(
    pob.loc[
        pob["ÁREA GEOGRÁFICA"].eq("Total")
        & pob["poblacion"].eq(0),
        ["MPIO", "DPMP", "anio", "poblacion"],
    ]
)

REGISTROS POR AÑO


,registros,codigos_territoriales,areas
anio,,,
2018,3369,1123,3
2019,3369,1123,3
2020,3369,1123,3
2021,3369,1123,3
2022,3369,1123,3
2023,3369,1123,3
2024,3369,1123,3
2025,3369,1123,3
2026,3369,1123,3


REGISTROS POR ÁREA


,registros
ÁREA GEOGRÁFICA,
Cabecera Municipal,28075
Centros Poblados y Rural Disperso,28075
Total,28075


RANGOS DE POBLACIÓN


,count,min,max
ÁREA GEOGRÁFICA,,,
Cabecera Municipal,28075,0,7914911
Centros Poblados y Rural Disperso,28075,0,242076
Total,28075,0,7945996


CEROS POR ÁREA


,registros
ÁREA GEOGRÁFICA,
Cabecera Municipal,483
Centros Poblados y Rural Disperso,29
Total,29


POBLACIÓN TOTAL EN CERO


,MPIO,DPMP,anio,poblacion
1788,27493,Nuevo Belén de Bajirá,2018,0
5157,27493,Nuevo Belén de Bajirá,2019,0
8526,27493,Nuevo Belén de Bajirá,2020,0
10047,94663,Mapiripana (ANM),2020,0
11895,27493,Nuevo Belén de Bajirá,2021,0
13416,94663,Mapiripana (ANM),2021,0
15264,27493,Nuevo Belén de Bajirá,2022,0
16785,94663,Mapiripana (ANM),2022,0
18633,27493,Nuevo Belén de Bajirá,2023,0
20154,94663,Mapiripana (ANM),2023,0


In [7]:
if pob[claves].isna().any().any() or pob.duplicated(claves).any():
    raise ValueError(
        "Revisar claves incompletas o duplicadas antes de continuar."
    )

if not pob["ÁREA GEOGRÁFICA"].isin(areas_esperadas).all():
    raise ValueError(
        "Revisar categorías de área antes de continuar."
    )

pob_anual = (
    pob.pivot(
        index=["codigo_municipio", "anio"],
        columns="ÁREA GEOGRÁFICA",
        values="poblacion",
    )
    .reindex(columns=areas_esperadas)
    .rename(columns={
        "Cabecera Municipal": "poblacion_cabecera",
        "Centros Poblados y Rural Disperso": "poblacion_centros_rural",
        "Total": "poblacion_total",
    })
    .reset_index()
)

pob_anual.columns.name = None

partes = [
    "poblacion_cabecera",
    "poblacion_centros_rural",
]

pob_anual["suma_areas"] = (
    pob_anual[partes].sum(axis=1, min_count=2)
)

pob_anual["diferencia"] = (
    pob_anual["poblacion_total"]
    - pob_anual["suma_areas"]
)

inconsistente = (
    pob_anual[partes + ["poblacion_total"]]
    .isna().any(axis=1)
    | pob_anual["diferencia"].ne(0)
)

print(
    "Territorios/años inconsistentes:",
    inconsistente.sum(),
)

display(pob_anual.loc[inconsistente])

# Incorporar nombres y departamento a la tabla auxiliar.
catalogo_pob = pob[
    ["codigo_municipio", "codigo_departamento", "DPMP", "DPNOM"]
].drop_duplicates()

pob_anual = pob_anual.merge(
    catalogo_pob,
    on="codigo_municipio",
    how="left",
    validate="many_to_one",
)

# No dividir entre población cero.
denominador = pob_anual["poblacion_total"].where(
    pob_anual["poblacion_total"].gt(0)
)

pob_anual["porcentaje_centros_rural"] = (
    pob_anual["poblacion_centros_rural"]
    / denominador * 100
).round(2)

display(pob_anual.head())

Territorios/años inconsistentes: 0


,codigo_municipio,anio,poblacion_cabecera,poblacion_centros_rural,poblacion_total,suma_areas,diferencia


,codigo_municipio,anio,poblacion_cabecera,poblacion_centros_rural,poblacion_total,suma_areas,diferencia,codigo_departamento,DPMP,DPNOM,porcentaje_centros_rural
0,05001,2018,2382405,44728,2427133,2427133,0,05,Medellín,Antioquia,1.84
1,05001,2019,2409046,46290,2455336,2455336,0,05,Medellín,Antioquia,1.89
2,05001,2020,2441882,48073,2489955,2489955,0,05,Medellín,Antioquia,1.93
3,05001,2021,2459249,48371,2507620,2507620,0,05,Medellín,Antioquia,1.93
4,05001,2022,2466239,48470,2514709,2514709,0,05,Medellín,Antioquia,1.93


In [8]:
resumen_pob_anual = (
    pob_anual.groupby("anio").agg(
        codigos_territoriales=("codigo_municipio", "nunique"),
        poblacion_total=("poblacion_total", "sum"),
        poblacion_cabecera=("poblacion_cabecera", "sum"),
        poblacion_centros_rural=("poblacion_centros_rural", "sum"),
    )
)

display(resumen_pob_anual)

print("CASOS TERRITORIALES QUE REQUIEREN REVISIÓN")

display(
    pob_anual.loc[
        pob_anual["codigo_municipio"].isin(
            ["27493", "27615", "94343", "94663"]
        )
        & pob_anual["anio"].between(2018, 2024),
        [
            "codigo_municipio",
            "DPMP",
            "anio",
            "poblacion_total",
        ],
    ].sort_values(["codigo_municipio", "anio"])
)

,codigos_territoriales,poblacion_total,poblacion_cabecera,poblacion_centros_rural
anio,,,,
2018,1123,48258494,36414521,11843973
2019,1123,49266526,37219289,12047237
2020,1123,50390790,38130362,12260428
2021,1123,51115637,38710371,12405266
2022,1123,51643565,39130235,12513330
2023,1123,52117067,39507801,12609266
2024,1123,52613753,39903699,12710054
2025,1123,53057212,40257670,12799542
2026,1123,53399171,40529928,12869243


CASOS TERRITORIALES QUE REQUIEREN REVISIÓN


,codigo_municipio,DPMP,anio,poblacion_total
14875,27493,Nuevo Belén de Bajirá,2018,0
14876,27493,Nuevo Belén de Bajirá,2019,0
14877,27493,Nuevo Belén de Bajirá,2020,0
14878,27493,Nuevo Belén de Bajirá,2021,0
14879,27493,Nuevo Belén de Bajirá,2022,0
14880,27493,Nuevo Belén de Bajirá,2023,0
14881,27493,Nuevo Belén de Bajirá,2024,29812
14975,27615,Riosucio,2018,53441
14976,27615,Riosucio,2019,55472
14977,27615,Riosucio,2020,57468


In [9]:
divipola = pd.read_csv(
    bronze / "DIVIPOLA-_C_digos_municipios.csv",
    encoding="utf-8-sig",
    dtype="string",
)

divipola["codigo_municipio"] = normalizar_codigo(
    divipola["Código Municipio"], 5
)

divipola["codigo_departamento_div"] = normalizar_codigo(
    divipola["Código Departamento"], 2
)

if (
    divipola["codigo_municipio"].isna().any()
    or divipola["codigo_municipio"].duplicated().any()
):
    raise ValueError("Revisar claves de DIVIPOLA.")

campos_div = [
    "codigo_municipio",
    "codigo_departamento_div",
    "Nombre Municipio",
    "Nombre Departamento",
    "Tipo: Municipio / Isla / Área no municipalizada",
]

cruce_div = pob_anual.merge(
    divipola[campos_div],
    on="codigo_municipio",
    how="left",
    validate="many_to_one",
    indicator=True,
)

display(
    cruce_div.groupby("_merge", observed=True).agg(
        registros=("anio", "size"),
        codigos=("codigo_municipio", "nunique"),
    )
)

print("CÓDIGOS SIN COINCIDENCIA")
display(
    cruce_div.loc[
        cruce_div["_merge"].eq("left_only"),
        ["codigo_municipio", "DPMP", "DPNOM"],
    ].drop_duplicates()
)

print("DEPARTAMENTOS DIFERENTES ENTRE CÓDIGOS COINCIDENTES")
display(
    cruce_div.loc[
        cruce_div["_merge"].eq("both")
        & cruce_div["codigo_departamento"].ne(
            cruce_div["codigo_departamento_div"]
        ).fillna(False),
        [
            "codigo_municipio",
            "DPMP",
            "codigo_departamento",
            "codigo_departamento_div",
        ],
    ].drop_duplicates()
)

,registros,codigos
_merge,,
left_only,25,1
both,28050,1122


CÓDIGOS SIN COINCIDENCIA


,codigo_municipio,DPMP,DPNOM
27550,94663,Mapiripana (ANM),Guainía


DEPARTAMENTOS DIFERENTES ENTRE CÓDIGOS COINCIDENTES


,codigo_municipio,DPMP,codigo_departamento,codigo_departamento_div


In [10]:
archivo_men = (
    bronze
    / "MEN_ESTADISTICAS_EN_EDUCACION_EN_PREESCOLAR__B_SICA_Y_MEDIA_POR_MUNICIPIO.csv"
)

men = pd.read_csv(
    archivo_men,
    encoding="utf-8-sig",
    dtype="string",
    keep_default_na=False,
    na_values=[""],
)

men["codigo_municipio"] = normalizar_codigo(
    men["CÓDIGO_MUNICIPIO"], 5
)

men["anio"] = pd.to_numeric(
    men["AÑO"], errors="coerce"
)

# Identificar y separar el nivel nacional.
es_nacional = (
    men["CÓDIGO_MUNICIPIO"]
    .str.strip()
    .str.fullmatch(r"0+", na=False)
)

print("REGISTROS NACIONALES")
display(
    men.loc[es_nacional, ["AÑO", "MUNICIPIO"]]
)

men_periodo = men.loc[
    men["anio"].between(2018, 2024)
    & ~es_nacional
].copy()

cruce_men = men_periodo.merge(
    pob_anual,
    on=["codigo_municipio", "anio"],
    how="left",
    validate="one_to_one",
    indicator=True,
)

display(
    cruce_men["_merge"]
    .value_counts()
    .to_frame("registros")
)

print("SIN POBLACIÓN ENCONTRADA")
display(
    cruce_men.loc[
        cruce_men["_merge"].eq("left_only"),
        ["AÑO", "CÓDIGO_MUNICIPIO", "MUNICIPIO"],
    ]
)

print(
    "Población encontrada pero no positiva:",
    cruce_men["poblacion_total"].le(0).sum(),
)

REGISTROS NACIONALES


,AÑO,MUNICIPIO
1269,2019,NACIONAL
1665,2020,NACIONAL
2085,2021,NACIONAL


,registros
_merge,
both,7850
left_only,0
right_only,0


SIN POBLACIÓN ENCONTRADA


,AÑO,CÓDIGO_MUNICIPIO,MUNICIPIO


Población encontrada pero no positiva: 0


In [11]:
partes_internet = []

lector_internet = pd.read_csv(
    bronze / "ACCESOS_INTERNET_FIJO_2_8.csv",
    sep=";",
    encoding="utf-8",
    usecols=[
        "ANNO",
        "TRIMESTRE",
        "ID_MUNICIPIO",
        "ACCESOS",
    ],
    dtype="string",
    chunksize=100_000,
    keep_default_na=False,
    na_values=[""],
)

for bloque in lector_internet:
    bloque["anio"] = pd.to_numeric(
        bloque["ANNO"], errors="coerce"
    )

    bloque["trimestre"] = pd.to_numeric(
        bloque["TRIMESTRE"], errors="coerce"
    )

    bloque = bloque.loc[
        bloque["anio"].between(2018, 2024)
        & bloque["trimestre"].eq(4)
    ].copy()

    bloque["codigo_municipio"] = normalizar_codigo(
        bloque["ID_MUNICIPIO"], 5
    )

    bloque["accesos"] = pd.to_numeric(
        bloque["ACCESOS"], errors="coerce"
    )

    if (
        bloque["codigo_municipio"].isna().any()
        or bloque["accesos"].isna().any()
        or bloque["accesos"].lt(0).any()
        or bloque["accesos"].mod(1).ne(0).any()
    ):
        raise ValueError(
            "Revisar claves o accesos inválidos en internet T4."
        )

    partes_internet.append(
        bloque.groupby(
            ["codigo_municipio", "anio"]
        )["accesos"]
        .sum()
        .reset_index()
    )

internet_t4 = (
    pd.concat(partes_internet, ignore_index=True)
    .groupby(
        ["codigo_municipio", "anio"],
        as_index=False,
    )["accesos"]
    .sum()
    .rename(columns={"accesos": "accesos_t4"})
)

del partes_internet

# Revisar también los códigos de internet sin población.
cruce_internet = internet_t4.merge(
    pob_anual,
    on=["codigo_municipio", "anio"],
    how="left",
    validate="one_to_one",
    indicator=True,
)

print("INTERNET T4 SIN POBLACIÓN ENCONTRADA")
display(
    cruce_internet.loc[
        cruce_internet["_merge"].eq("left_only"),
        ["codigo_municipio", "anio", "accesos_t4"],
    ]
)

# Comparación exploratoria de las tres fuentes.
comparacion = (
    cruce_men.drop(columns="_merge")
    .merge(
        internet_t4,
        on=["codigo_municipio", "anio"],
        how="left",
        validate="one_to_one",
        indicator="estado_internet",
    )
)

denominador = comparacion["poblacion_total"].where(
    comparacion["poblacion_total"].gt(0)
)

comparacion["accesos_por_100_habitantes"] = (
    comparacion["accesos_t4"]
    / denominador * 100
).round(2)

print("DISPONIBILIDAD DE INTERNET T4 EN EDUCACIÓN")
display(
    comparacion["estado_internet"]
    .value_counts()
    .to_frame("registros")
)

print("REGISTROS SIN INTERNET T4")
display(
    comparacion.loc[
        comparacion["estado_internet"].eq("left_only"),
        ["codigo_municipio", "MUNICIPIO", "anio"],
    ]
)

print("EJEMPLO: MEDELLÍN 2024")
display(
    comparacion.loc[
        comparacion["codigo_municipio"].eq("05001")
        & comparacion["anio"].eq(2024),
        [
            "codigo_municipio",
            "anio",
            "poblacion_total",
            "accesos_t4",
            "accesos_por_100_habitantes",
        ],
    ]
)

INTERNET T4 SIN POBLACIÓN ENCONTRADA


,codigo_municipio,anio,accesos_t4
4067,27086,2018,2
4068,27086,2019,4


DISPONIBILIDAD DE INTERNET T4 EN EDUCACIÓN


,registros
estado_internet,
both,7813
left_only,37
right_only,0


REGISTROS SIN INTERNET T4


,codigo_municipio,MUNICIPIO,anio
1095,91430,La Victoria,2023
1096,91460,Miriti - Paraná,2023
1100,91669,Puerto Santander,2023
1105,94884,Puerto Colombia,2023
1108,94887,Pana Pana,2023
2232,91407,La Pedrera,2018
2233,91430,La Victoria,2018
2234,91460,Miriti - Paraná,2018
2236,91536,Puerto Arica,2018
2238,91669,Puerto Santander,2018


EJEMPLO: MEDELLÍN 2024


,codigo_municipio,anio,poblacion_total,accesos_t4,accesos_por_100_habitantes
6739,05001,2024,2524747,791436,31.35
